# SynthMorph — Variant A (Baseline sm-shapes) on Kaggle

**GPU:** T4 / P100 (~16GB VRAM) — dùng `--nb-features 256` (chuẩn paper)

| Setting | Value |
|---|---|
| Generator | `baseline` (Variant A) |
| NB Features | 256 (paper standard) |
| Save every | 20 iters + best model |
| Output | `/kaggle/working/runs/` |

> **Lưu ý:** Sau khi session kết thúc, download file `checkpoint_best.pth` và `config.json`
> từ Kaggle Output tab để dùng ở local.

In [ ]:
# ============================================================
# CELL 1: Check GPU
# ============================================================
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

In [ ]:
# ============================================================
# CELL 2: Clone từ GitHub & Install dependencies
# ============================================================
import os
%cd /kaggle/working

# Xoá run cũ nếu resume
if not os.path.exists('SynthMorph'):
    !git clone https://github.com/YOUR_USERNAME/SynthMorph.git
else:
    print('[INFO] Code đã tồn tại. Pull update...')
    !cd SynthMorph && git pull

%cd SynthMorph
!pip install -q nibabel tqdm
print('\n✅ Setup complete')

In [ ]:
# ============================================================
# CELL 3: Dry-run (20 iters, verify everything works)
# ============================================================
RUNS_DIR = '/kaggle/working/runs'

!python train_sm_shapes.py \
    --generator-type baseline \
    --iters 20 \
    --nb-features 256 \
    --vis-every 5 \
    --save-every 10 \
    --log-every 1 \
    --runs-dir {RUNS_DIR} \
    --resume

In [ ]:
# ============================================================
# CELL 4: FULL TRAINING (400k iters)
# Save mỗi 20 iter + best model tự động
# ============================================================
TOTAL_ITERS  = 400_000
NB_FEATURES  = 256      # Paper standard, OK với T4 16GB VRAM
RUNS_DIR     = '/kaggle/working/runs'
VIS_EVERY    = 1000     # Lưu ảnh step1/2/3 mỗi 1000 iter
SAVE_EVERY   = 20       # Save checkpoint mỗi 20 iter

!python train_sm_shapes.py \
    --generator-type baseline \
    --iters {TOTAL_ITERS} \
    --nb-features {NB_FEATURES} \
    --vis-every {VIS_EVERY} \
    --save-every {SAVE_EVERY} \
    --log-every 100 \
    --runs-dir {RUNS_DIR} \
    --resume

In [ ]:
# ============================================================
# CELL 5: Xem ảnh trực quan hóa (step1/2/3)
# ============================================================
import glob
from IPython.display import display, Image

all_runs = sorted(glob.glob('/kaggle/working/runs/sm_shapes_baseline_*'))
if all_runs:
    latest_run = all_runs[-1]
    print(f'Latest run: {latest_run}')
    
    for step in ['step1_labels', 'step2_mri', 'step3_registration']:
        imgs = sorted(glob.glob(f'{latest_run}/vis/{step}_*.png'))
        if imgs:
            print(f'\n--- {step} (latest) ---')
            display(Image(imgs[-1]))

In [ ]:
# ============================================================
# CELL 6: Compress outputs để download
# ============================================================
import shutil, glob

latest_run = sorted(glob.glob('/kaggle/working/runs/sm_shapes_baseline_*'))[-1]
out_zip = '/kaggle/working/variant_a_baseline_checkpoint'

shutil.make_archive(out_zip, 'zip', latest_run)
import os
size_mb = os.path.getsize(out_zip + '.zip') / 1e6
print(f'✅ Saved {out_zip}.zip ({size_mb:.1f} MB)')
print('Download từ Kaggle Output tab →')